In [ ]:
import subprocess, os
from pathlib import Path
import pandas as pd
from IPython.display import display

def run_tshark_count(pcap_path: Path, display_filter: str) -> int:
    cmd = [
        "tshark", "-r", str(pcap_path),
        "-Y", display_filter,
        "-T", "fields", "-e", "frame.number"
    ]
    result = subprocess.run(cmd, stdout=subprocess.PIPE,
                            stderr=subprocess.PIPE, text=True, check=True)
    return len(result.stdout.strip().splitlines())

cellreplay_dirs = [
    r"" # Path for CSV files
]

mahimahi_dirs = [
    r"" # Path for CSV files
]

mahimahi_4g_dirs = [
    r"" # Path for CSV files
]

client_ip = "10.0.16.2"

def collect_records(directories):
    recs = []
    for folder in directories:
        if not os.path.exists(folder):
            continue
        for file in os.listdir(folder):
            if file.lower().endswith((".pcap", ".pcapng")):
                pcap_path = Path(folder) / file
                try:
                    retrans = run_tshark_count(pcap_path, "tcp.analysis.retransmission")
                    total   = run_tshark_count(pcap_path, "tcp")
                    perc    = (retrans / total * 100) if total else 0
                    recs.append({
                        "Algorithm"         : Path(file).stem,
                        "Retransmissions"   : retrans,
                        "Total TCP Packets" : total,
                        "Retrans %"         : perc,
                    })
                except subprocess.CalledProcessError:
                    pass
    return pd.DataFrame(recs)

def summarise(df: pd.DataFrame) -> pd.DataFrame:
    return (df
            .groupby("Algorithm", as_index=False)
            .agg({
                "Retransmissions": "mean",
                "Total TCP Packets": "mean",
                "Retrans %": "mean"
            })
            .round(6)
            .rename(columns={
                "Retransmissions": "Avg Retransmissions",
                "Total TCP Packets": "Avg Total TCP Packets",
                "Retrans %": "Avg Retrans %"
            }))

summary_cellreplay = summarise(collect_records(cellreplay_dirs))
summary_mahimahi   = summarise(collect_records(mahimahi_dirs))
summary_4g         = summarise(collect_records(mahimahi_4g_dirs))

print("CellReplay (5G)")
display(summary_cellreplay)

print(" Mahimahi (5G)")
display(summary_mahimahi)

print(" Mahimahi (4G)")
display(summary_4g)



=== CellReplay (5G) ===


,Algorithm,Avg Retransmissions,Avg Total TCP Packets,Avg Retrans %
0,BOLA,2.00,238397.00,0.000839
1,Gelato,1.75,249807.50,0.000700
2,LinearBBA,0.75,248004.25,0.000302
3,MPC,2.25,238480.50,0.000940
4,Pensieve,1.50,251827.50,0.000593
5,TTP,1.50,238240.75,0.000625



=== Mahimahi (5G) ===


,Algorithm,Avg Retransmissions,Avg Total TCP Packets,Avg Retrans %
0,BOLA,2.50,222277.00,0.001086
1,Gelato,3.00,233326.75,0.001244
2,LinearBBA,2.00,231356.00,0.000839
3,MPC,2.25,223014.25,0.000971
4,Pensieve,4.50,232681.00,0.001889
5,TTP,2.00,222836.25,0.000880



=== Mahimahi (4G) ===


,Algorithm,Avg Retransmissions,Avg Total TCP Packets,Avg Retrans %
0,BOLA,131.666667,170638.333333,0.143533
1,Gelato,49.333333,171837.666667,0.029096
2,LinearBBA,53.000000,181782.333333,0.030106
3,MPC,45.000000,173200.333333,0.025491
4,Pensieve,67.666667,147693.000000,0.074226
5,TTP,49.666667,179226.000000,0.029802
